In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Fri Mar  7 16:34:17 2025

   Copyright 2025 Sylvan Energy Analytics

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

"""


import pandas as pd
import numpy as np
import os

project = 'Solar_NoLTFrights'
Utility_POD = 'BPAT.PGE'

# create output directories
if os.path.exists('outputs') == False:
    os.mkdir('outputs')

# read input data
project_information = pd.read_csv(os.path.join('inputs','project_info.csv'),index_col='project')
ltf_rights = pd.read_csv(os.path.join('inputs','Utility_LTF_rights.csv'))
ptdfs = pd.read_csv(os.path.join('inputs','PTDFs.csv'))
# The list of flowgates
path_list = list(np.unique(ptdfs['Path']))

# The list of flowgates with historical flow data
path_list_with_data = []
# `flow_data` is a list of pandas dataframes. Important columns: year,month,HE,actual flow,headroom,min(TTC,SOL)
flow_data = []
# A list of path allocation factors. Calculated from line 63 to 72
path_allocation_factors = []
# This loop below mainly calculate the "path_allocation_factor" for each path/flowgate, where "path_allocation_factor" means how much percentage of TTC is used for the  (PGE) on a specific path/flowgate on average. It also calcualte the headroom of each row. 
for path in path_list:
    # If the path has historical flow data
    if os.path.exists(os.path.join('inputs','historical_flows',path+'.csv')):
        # Record path/flowgate with data
        path_list_with_data.append(path)
        # Read historical flow data
        flow_data_tmp = pd.read_csv(os.path.join('inputs','historical_flows',path+'.csv'))
        
        # calculate the maximum flow as the minimum of the TTC and SOL
        flow_data_tmp['TTC'].fillna(999999,inplace=True)
        flow_data_tmp['SOL'].fillna(999999,inplace=True)
        # Instead of doing a interpolation, Elaine filled the missing actual flow with 999999, and later in line 77, she just ignored the rows with no 'actual flow'
        flow_data_tmp['actual flow'].fillna(999999,inplace=True)
        flow_data_tmp['min(TTC,SOL)'] = np.minimum(flow_data_tmp['TTC'],flow_data_tmp['SOL'])
        
        # calculate average TTC. `TTC_average` is later used in line 69
        TTC_average = np.mean(flow_data_tmp['min(TTC,SOL)'])
        
        # calculate utility path allocation factor (capping between 0 and 1)
        # path allocation factor here means how much of a path's capacity is effectively reserved for the utility's use
        path_allocation_factor_tmp = 0
        # sum(PTDF * LTF rights) / TTC_average. It means how much percentage of TTC is used for the utility on this path/flowgate on average.
        for _, row in ltf_rights.iterrows():
            # Find the cooreponding PTDF given flowgate, POR, POD
            path_allocation_factor_tmp += ptdfs[(ptdfs['Path'] == path) & (ptdfs['POR']==row['POR']) & (ptdfs['POD']==row['POD'])]['PTDF'].iloc[0]*row['LTF rights (MW)']/TTC_average
        # caps the factor at 1
        path_allocation_factors.append(max(min(path_allocation_factor_tmp,1),0))
        
        # calculate headroom
        flow_data_tmp['headroom'] = flow_data_tmp['min(TTC,SOL)'] - flow_data_tmp['actual flow']
        
        # exclude timepoints for which TTC and SOL or actual flow data is unavailable. Elaine's approach is differnt from mine. She didn't do an interpolation here
        flow_data_tmp = flow_data_tmp[(flow_data_tmp['min(TTC,SOL)'] != 999999) & (flow_data_tmp['min(TTC,SOL)'] != 0) & (flow_data_tmp['actual flow'] != 999999) & (flow_data_tmp['actual flow'] != 0)]
        flow_data.append(flow_data_tmp)
    
# Estimate impacts of delivering output to POD on flows across each path based on PTDFs
project_info = project_information.loc[project]
project_hourly_data = pd.read_csv(os.path.join('inputs','project_hourly_data',project_info['Hourly data']))
# "deliverable output" here means the output that is not at risk
project_hourly_data['deliverable output (MW)'] = np.minimum(project_hourly_data['total output (MW)'],project_hourly_data['available LTF tx (MW)'])
project_hourly_data['output at risk (MW)'] = project_hourly_data['total output (MW)'] - project_hourly_data['deliverable output (MW)']

# For each flowgate, add a new column 'path_flow_impact' := short-term capacity * PTDF. It means how much short-term capacity for the current project will flow through a path/flowgate
for path in path_list_with_data:
    project_hourly_data[path+'_flow_impact'] = project_hourly_data['output at risk (MW)']*np.array(ptdfs[(ptdfs['POR']==project_info['POI or POD']) & (ptdfs['POD']==Utility_POD) & (ptdfs['Path']==path)]['PTDF'])

# initialize columns to record curtailment probabilit and expected curtailment
project_hourly_data['curtailment probability'] = project_hourly_data['output at risk (MW)']*0
project_hourly_data['expected curtailment (MW)'] = project_hourly_data['output at risk (MW)']*0

# calculate probability of curtailment and average curtailment in each hour with project hourly data based on month-hour headroom distributions
curtailment_prob_tmp = np.zeros(len(project_hourly_data))
curtailment_exp_tmp = np.zeros(len(project_hourly_data))
for HE in range(1, 25):
    for month in range(1, 13):
        total_congestion_probability = 0
        total_average_curtailment = 0
        
        # pull path flow impacts for month-hour bin
        project_hourly_subset = project_hourly_data[(project_hourly_data['month']==month) & (project_hourly_data['HE']==HE)]
        
        for p_ind in range(len(path_list_with_data)):
            # Get the name of the path/flowgate
            path = path_list_with_data[p_ind]
            
            # pull estimated path flow impacts for month-hour bin
            flow_impact = [project_hourly_subset[path+'_flow_impact']]

            # estimate utility headroom for month-hour bin
            flow_data_subset = flow_data[p_ind][(flow_data[p_ind]['month']==month) & (flow_data[p_ind]['HE']==HE)]
            # `utility_headroom` is an array with headroom for a specific utility in a given path/flowgate
            utility_headroom = np.transpose([path_allocation_factors[p_ind]*np.maximum(0,flow_data_subset['min(TTC,SOL)'] - flow_data_subset['actual flow'])])
            
            # estimate the curtailed flow for each combination of project hourly output data and historical headroom data
            # flow_impact is an array. np.ones(np.shape(utility_headroom)) is another array with 1s. The result's shape is (len(flow_impact), len(utility_headroom)), where each row is a copy of flow_impact[i]
            flow_impact_matrix = np.outer(flow_impact,np.ones(np.shape(utility_headroom)))
            # shape: (len(flow_impact), len(utility_headroom)). Each column is a copy of utility_headroom[j]
            utility_headroom_matrix = np.outer(np.ones(np.shape(flow_impact)),utility_headroom)            
            actual_flow_matrix = np.minimum(flow_impact_matrix,utility_headroom_matrix)
            curtailed_flow = flow_impact_matrix - actual_flow_matrix
            
            # calculate the project curtailment required to achieve the curtailed flow based on the corresponding PTDF
            curtailed_output = curtailed_flow/(np.array(ptdfs[(ptdfs['POR']==project_info['POI or POD']) & (ptdfs['POD']==Utility_POD) & (ptdfs['Path']==path)]['PTDF'])[0])
            
            # update the likelihood of curtailment and average curtailment for each hour with hourly project data
            # this approach assumes curtailments due to constraints on different paths are non-overlapping (i.e. could overestimate curtailment)
            curtailment_prob_tmp[project_hourly_subset.index] += np.mean(curtailed_output > 0,axis=1)
            curtailment_exp_tmp[project_hourly_subset.index] += np.mean(curtailed_output,axis=1)


project_hourly_data['curtailment probability'] = curtailment_prob_tmp
project_hourly_data['expected curtailment (MW)'] = curtailment_exp_tmp
# this is called "delivered" in my script
project_hourly_data['derated output (MW)'] = np.maximum(project_hourly_data['total output (MW)'] -project_hourly_data['expected curtailment (MW)'],0)

project_hourly_data.to_csv(os.path.join('outputs',project+'_results.csv'),index=False)

In [ ]:
projects = ['Solar_MinLTFrights', 'Solar_NoLTFrights', 'Wind_MinLTFrights', 'Wind_NoLTFrights']
for project in projects:
    df = pd.read_csv(f"outputs/{project}_results.csv")
    delivered_percentage = (df['derated output (MW)'].sum() / df["total output (MW)"].sum()) * 100
    print(f"{project}: {delivered_percentage:.2f}%")

Solar_MinLTFrights: 99.18%

Solar_NoLTFrights: 91.39%

Wind_MinLTFrights: 97.73%

Wind_NoLTFrights: 96.12%

## Get the 12 by 24 constrain table for each project

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os


def plot_curtailment_heatmap(
    curtailment_df: pd.DataFrame, title="Curtailment Rate Heatmap"
):
    curtailment_df_percentage = curtailment_df * 100
    plt.figure(figsize=(14, 8))
    sns.heatmap(
        curtailment_df_percentage,
        cmap="YlOrRd",
        linewidths=0.5,
        annot=True,
        fmt=".0f",
    )
    for text in plt.gca().texts:
        text.set_text(f"{text.get_text()}%")
    plt.title(title, fontsize=16)
    plt.ylabel("Hour Ending (HE)")
    plt.xlabel("Month")
    plt.gca().set_aspect(0.5, adjustable="box")
    plt.tight_layout()
    plt.show()


def process_curtailment_data(file_path):
    """
    Loads and processes the curtailment data from the given file path.

    Parameters:
        file_path (str): Path to the CSV file containing curtailment data.

    Returns:
        pivot_table (DataFrame): Pivot table with curtailment rates for plotting heatmap.
    """
    df = pd.read_csv(file_path)
    grouped = (
        df.groupby(["month", "HE"])[["total output (MW)", "expected curtailment (MW)"]]
        .sum()
        .reset_index()
    )
    grouped["curtailment_rate"] = np.where(
        grouped["total output (MW)"] == 0,
        0,
        grouped["expected curtailment (MW)"] / grouped["total output (MW)"],
    )
    pivot_table = grouped.pivot(index="HE", columns="month", values="curtailment_rate")
    return pivot_table

In [ ]:
file_path = os.path.join("outputs", "Solar_MinLTFrights_results.csv")
pivot_table = process_curtailment_data(file_path)
plot_curtailment_heatmap(pivot_table, "Solar_MinLTF")

In [ ]:
file_path = os.path.join("outputs", "Solar_NoLTFrights_results.csv")
pivot_table = process_curtailment_data(file_path)
plot_curtailment_heatmap(pivot_table, "Solar_NoLTF")

In [ ]:
file_path = os.path.join("outputs", "Wind_MinLTFrights_results.csv")
pivot_table = process_curtailment_data(file_path)
plot_curtailment_heatmap(pivot_table, "Wind_MinLTF")

In [ ]:
file_path = os.path.join("outputs", "Wind_NoLTFrights_results.csv")
pivot_table = process_curtailment_data(file_path)
plot_curtailment_heatmap(pivot_table, "Wind_NoLTF")